In [1]:
# --------------------------------------------------
# 1. Imports (add these to the existing ones)
# --------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools
from torchvision.transforms import Resize
from ast import literal_eval
from sklearn.metrics import matthews_corrcoef
from torchvision import models
from transformers import AutoTokenizer, AutoModelForMaskedLM
import math
import json
from collections import defaultdict
import seaborn as sns
from matplotlib.colors import LogNorm
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
# --------------------------------------------------

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [2]:
DEVICE    = 'cuda:0' if torch.cuda.is_available() else 'cpu'

this is for all_endpoint + non_functional

In [3]:
with open('cleaned_clustered_dataset.json') as f:
    clustered_dataset = json.load(f)



In [4]:
class_count = defaultdict(int)
for cluster in clustered_dataset:
    for s in cluster:
        for c in s['classes']:
            class_count[c] += 1

class_count

defaultdict(int,
            {'other-functional': 7040,
             'metabolic': 11589,
             'inhibitor': 2046,
             'toxic': 22440,
             'anti-cancer': 12390,
             'drug-delivery': 2202,
             'anti-bacterial': 29969,
             'anti-fungal': 12747,
             'anti-parasitic': 6586,
             'anti-viral': 6894,
             'signal-peptide': 22226,
             'immunological': 4413,
             'cell-cell-communication': 3249,
             'non-functional': 163115})

In [5]:
# cleaning clusters , removing clustures with empty sequences or empty classes
removes_seqs = 0
for i, cluster in enumerate(clustered_dataset):
    new_cluster = []
    for entry in cluster:
        if entry['sequence'] == '' or len(entry['classes']) == 0:
            removes_seqs += 1
        else:
            new_cluster.append(entry)

    clustered_dataset[i] = new_cluster

unfiltured_clusters_size = len(clustered_dataset)
clustered_dataset = [cluster for cluster in clustered_dataset if len(cluster) > 0]
print(f"Removed {unfiltured_clusters_size - len(clustered_dataset)} clusters")
print(f"Removed {removes_seqs} sequences")

Removed 0 clusters
Removed 0 sequences


# dividing into train and test


In [6]:
train_val_test_ratio = [0.7, 0.1, 0.2]
random_state = 42

if len(train_val_test_ratio) != 3:
    raise ValueError("train_val_test_ratio must contain [train, val, test].")

ratio_sum = sum(train_val_test_ratio)
if ratio_sum <= 0:
    raise ValueError("Sum of train_val_test_ratio must be > 0.")

train_ratio, val_ratio, test_ratio = [r / ratio_sum for r in train_val_test_ratio]

all_seqs_count = sum(len(cluster) for cluster in clustered_dataset)
print('all_seqs_count', all_seqs_count)

train_size = int(all_seqs_count * train_ratio)
val_size = int(all_seqs_count * val_ratio)
test_size = all_seqs_count - train_size - val_size

train_seqs = []
val_seqs = []
test_seqs = []

train_cluster_count = 0
val_cluster_count = 0
test_cluster_count = 0

all_cluster_ids = [i for i in range(len(clustered_dataset))]
rng = random.Random(random_state)
rng.shuffle(all_cluster_ids)

for cluster_id in all_cluster_ids:
    cluster = clustered_dataset[cluster_id]
    if len(train_seqs) < train_size:
        train_seqs.extend(cluster)
        train_cluster_count += 1
    elif len(val_seqs) < val_size:
        val_seqs.extend(cluster)
        val_cluster_count += 1
    else:
        test_seqs.extend(cluster)
        test_cluster_count += 1

print('train_seqs', len(train_seqs))
print('val_seqs', len(val_seqs))
print('test_seqs', len(test_seqs))

print('train_cluster_count', train_cluster_count)
print('val_cluster_count', val_cluster_count)
print('test_cluster_count', test_cluster_count)

all_seqs_count 257812
train_seqs 180471
val_seqs 25781
test_seqs 51560
train_cluster_count 87893
val_cluster_count 12557
test_cluster_count 25276


# creating dataset

In [7]:
# other-functional is present with known functionl class then remove other-functional
for item in train_seqs:
    if 'other-functional' in item['classes'] and len(item['classes']) > 1:
        item['classes'].remove('other-functional')
        if len(item['classes']) == 0: raise Exception("no class left")

for item in test_seqs:
    if 'other-functional' in item['classes'] and len(item['classes']) > 1:
        item['classes'].remove('other-functional')
        if len(item['classes']) == 0: raise Exception("no class left")

for item in val_seqs:
    if 'other-functional' in item['classes'] and len(item['classes']) > 1:
        item['classes'].remove('other-functional')
        if len(item['classes']) == 0: raise Exception("no class left")



In [8]:
all_endpoints = set([c for e in train_seqs+test_seqs+val_seqs for c in e['classes']]) 

sorted(list(all_endpoints))

['anti-bacterial',
 'anti-cancer',
 'anti-fungal',
 'anti-parasitic',
 'anti-viral',
 'cell-cell-communication',
 'drug-delivery',
 'immunological',
 'inhibitor',
 'metabolic',
 'non-functional',
 'other-functional',
 'signal-peptide',
 'toxic']

In [9]:
# endpoints = [
#  'anti-cancer',
#  'anti-fungal',
#  'anti-parasitic',
#  'anti-viral',
#  'non-functional'
#  ]

endpoints = ['anti-bacterial',
 'anti-cancer',
 'anti-fungal',
 'anti-parasitic',
 'anti-viral',
 'cell-cell-communication',
 'drug-delivery',
 'immunological',
 'inhibitor',
 'metabolic',
 'non-functional',
 'other-functional',
 'signal-peptide',
 'toxic']

 
endpoints_set = set(endpoints)
endpoint_index = {endpoint: i for i, endpoint in enumerate(endpoints)}
index_endpoint = {i: endpoint for i, endpoint in enumerate(endpoints)}


In [10]:
endpoints_set

{'anti-bacterial',
 'anti-cancer',
 'anti-fungal',
 'anti-parasitic',
 'anti-viral',
 'cell-cell-communication',
 'drug-delivery',
 'immunological',
 'inhibitor',
 'metabolic',
 'non-functional',
 'other-functional',
 'signal-peptide',
 'toxic'}

In [11]:
train_df_rows = []
for entry in train_seqs:
    row = [False]*len(endpoints)
    if len(set(entry['classes']).intersection(endpoints_set)) > 0: 
        
        for c in entry['classes']:
            if c in endpoints:
                row[endpoint_index[c]] = True

    
        row = [entry['sequence']] + row
        train_df_rows.append(row)
    
val_df_rows = []
for entry in val_seqs:
    row = [False]*len(endpoints)
    if len(set(entry['classes']).intersection(endpoints_set)) > 0: 

        for c in entry['classes']:
            if c in endpoints:
                row[endpoint_index[c]] = True

        row = [entry['sequence']] + row
        val_df_rows.append(row)

test_df_rows = []
for entry in test_seqs:
    row = [False]*len(endpoints)
    if len(set(entry['classes']).intersection(endpoints_set)) > 0: 

        for c in entry['classes']:
            if c in endpoints:
                row[endpoint_index[c]] = True

        row = [entry['sequence']] + row
        test_df_rows.append(row)

train_df = pd.DataFrame(train_df_rows, columns=['sequence'] + endpoints)
val_df = pd.DataFrame(val_df_rows, columns=['sequence'] + endpoints)
test_df = pd.DataFrame(test_df_rows, columns=['sequence'] + endpoints)


In [12]:
# # saving train, val, test dataset for classification
# train_df.to_csv('train_classification.csv')
# val_df.to_csv('validation_classification.csv')
# test_df.to_csv('test_classification.csv')

In [13]:
summary = train_df[endpoints].apply(pd.Series.value_counts)

# 2. Calculate True Percentage and add it as a new row
# We multiply by 100 to get a readable percentage
summary.loc['True %'] = train_df[endpoints].mean() * 100

# Print the final summary
print(summary)

        anti-bacterial    anti-cancer   anti-fungal  anti-parasitic  \
False    159502.000000  171656.000000  171554.00000   175865.000000   
True      20969.000000    8815.000000    8917.00000     4606.000000   
True %       11.619041       4.884441       4.94096        2.552211   

           anti-viral  cell-cell-communication  drug-delivery  immunological  \
False   175475.000000            178081.000000  178847.000000  177435.000000   
True      4996.000000              2390.000000    1624.000000    3036.000000   
True %       2.768312                 1.324312       0.899868       1.682265   

            inhibitor      metabolic  non-functional  other-functional  \
False   178988.000000  172233.000000    66678.000000     178237.000000   
True      1483.000000    8238.000000   113793.000000       2234.000000   
True %       0.821739       4.564722       63.053344          1.237872   

        signal-peptide          toxic  
False    164871.000000  164737.000000  
True      15600.0

In [14]:
summary = test_df[endpoints].apply(pd.Series.value_counts)

# 2. Calculate True Percentage and add it as a new row
# We multiply by 100 to get a readable percentage (e.g., 75.0 instead of 0.75)
summary.loc['True %'] = test_df[endpoints].mean() * 100

# Print the final summary
print(summary)

        anti-bacterial   anti-cancer  anti-fungal  anti-parasitic  \
False     45293.000000  49132.000000  49041.00000    50244.000000   
True       6267.000000   2428.000000   2519.00000     1316.000000   
True %       12.154771      4.709077      4.88557        2.552366   

          anti-viral  cell-cell-communication  drug-delivery  immunological  \
False   50281.000000              51014.00000   51168.000000   50648.000000   
True     1279.000000                546.00000     392.000000     912.000000   
True %      2.480605                  1.05896       0.760279       1.768813   

           inhibitor    metabolic  non-functional  other-functional  \
False   51191.000000  49450.00000    18796.000000      50873.000000   
True      369.000000   2110.00000    32764.000000        687.000000   
True %      0.715671      4.09232       63.545384          1.332428   

        signal-peptide         toxic  
False     47144.000000  47045.000000  
True       4416.000000   4515.000000  
True

In [15]:

esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")




In [16]:
class PeptideDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        label_columns,
        max_length: int = 128
    ):
        """
        Args:
            dataframe (pd.DataFrame): DataFrame containing the data.
                                      Must have a 'sequence' column.
            tokenizer (AutoTokenizer): A Hugging Face tokenizer (e.g., for ESM-2).
            label_columns (List[str]): A list of column names that represent the labels.
            max_length (int): Maximum sequence length for padding/truncation.
        """
        self.df = dataframe
        self.tokenizer = tokenizer
        self.sequences = self.df['sequence'].values
        self.labels = self.df[label_columns].values
        self.label_columns = label_columns
        self.max_length = max_length

        functional_idxs = []
        non_functional_idxs = []
        for i, label in enumerate(self.labels):
            if label[endpoint_index['non-functional']] == True:
                non_functional_idxs.append(i)
            else:
                functional_idxs.append(i)
        
        non_functional_idxs = [non_functional_idxs[i: i+1] for i in range(0, len(non_functional_idxs), 1)]
        non_functional_idxs = [i for i in non_functional_idxs]

        functional_idxs = [functional_idxs[i: i+1] for i in range(0, len(functional_idxs), 1)]
        self.all_idxs = non_functional_idxs + functional_idxs

        # Pre-tokenize ALL sequences once
        sequences = dataframe['sequence'].tolist()
        encodings = tokenizer(
            sequences,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        self.input_ids = encodings['input_ids']
        self.attention_masks = encodings['attention_mask']


    def __len__(self) -> int:
        """Returns the total number of samples in the dataset."""
        return len(self.all_idxs)

    def __getitem__(self, index: int):
        """
        Retrieves a single sample from the dataset.

        Args:
            index (int): The index of the sample to retrieve.

        Returns:
            A dictionary containing:
            - 'input_ids': Token IDs of the sequence.
            - 'attention_mask': Mask to avoid performing attention on padding tokens.
            - 'labels': A multi-hot encoded tensor of labels.
            - 'img_input': A dummy tensor to match the MultiModelNetwork's forward signature.
        """
        idx = random.choice(self.all_idxs[index])

        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'labels': torch.FloatTensor(self.labels[idx])
        }

In [17]:
LABEL_COLUMNS = endpoints

MAX_LEN = 100
train_dataset = PeptideDataset(train_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)
val_dataset = PeptideDataset(val_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)
test_dataset = PeptideDataset(test_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)

In [18]:
len(train_dataset),  len(val_dataset), len(test_dataset)

(180471, 25781, 51560)

In [19]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [20]:
# Calculate weights
NON_FUNC_IDX = endpoint_index['non-functional']
FUNC_INDICES = [i for k, i in endpoint_index.items() if k != 'non-functional']

In [21]:


class ESM2_Encoder(nn.Module):
    def __init__(self, model_name, trainable=True, unfreeze_last_n=0):
        super().__init__()
        self.esm_mlm = AutoModelForMaskedLM.from_pretrained(model_name)
        self.hidden_size = self.esm_mlm.config.hidden_size
        if not trainable:
            for param in self.esm_mlm.parameters():
                param.requires_grad = False
            if unfreeze_last_n > 0:
                for layer in self.esm_mlm.esm.encoder.layer[-unfreeze_last_n:]:
                    for param in layer.parameters():
                        param.requires_grad = True

    def forward(self, input_ids, attention_mask):
        return self.esm_mlm.esm(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

class SEBlock(nn.Module):
    """
    Mask-aware Squeeze-and-Excitation block.

    Input:
        x        : [B, C, L]
        seq_mask : [B, L], 1 for valid tokens, 0 for PAD

    The channel descriptor is computed using only valid sequence positions.
    """

    def __init__(self, channels, reduction=4):
        super().__init__()

        hidden = max(channels // reduction, 1)

        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid()
        )

    def forward(self, x, seq_mask):
        # [B, L] -> [B, 1, L]
        mask = seq_mask.unsqueeze(1).to(dtype=x.dtype)

        # Remove PAD contributions
        x_masked = x * mask

        # Number of valid positions per sequence
        lengths = seq_mask.sum(
            dim=1,
            keepdim=True
        ).clamp_min(1).to(dtype=x.dtype)

        # Masked global average pooling over sequence dimension
        # [B, C, L] -> [B, C]
        channel_descriptor = (
            x_masked.sum(dim=-1) / lengths
        )

        # Channel-wise gates
        gates = self.fc(channel_descriptor).unsqueeze(-1)

        # Apply channel recalibration
        return x * gates

class ConvBlock(nn.Module):
    """
    Mask-aware two-layer 1D convolution block.

    Uses LayerNorm instead of BatchNorm so that padded positions
    do not participate in batch/sequence normalization statistics.
    """

    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=False
        )

        self.norm1 = nn.LayerNorm(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=False
        )

        self.norm2 = nn.LayerNorm(out_channels)

    @staticmethod
    def apply_layernorm(x, norm):
        """
        Conv output: [B, C, L]
        LayerNorm expects normalized dimension at the end.
        """
        x = x.transpose(1, 2)      # [B, L, C]
        x = norm(x)
        x = x.transpose(1, 2)      # [B, C, L]
        return x

    def forward(self, x, mask):
        """
        x    : [B, C, L]
        mask : [B, 1, L]
        """
        # First convolution
        y = self.conv1(x)
        y = self.apply_layernorm(
            y,
            self.norm1
        )

        y = F.gelu(y)

        # Explicitly remove PAD activations
        y = y * mask

        # Second convolution
        y = self.conv2(y)

        y = self.apply_layernorm(
            y,
            self.norm2
        )

        y = F.gelu(y)

        # Explicitly remove PAD activations again
        y = y * mask

        return y

class EnhancedCNN1D(nn.Module):
    """
    Mask-aware multi-scale CNN for peptide/protein sequences.

    Branches:
        kernel 3 -> effective receptive field 5
        kernel 5 -> effective receptive field 9
        kernel 7 -> effective receptive field 13

    Each branch contains two convolutional layers.

    Output:
        [B, 2 * 3 * conv_dim]

    For conv_dim=128:
        output = [B, 768]
    """

    def __init__(
        self,
        vocab_size=33,
        embed_dim=128,
        conv_dim=128
    ):
        super().__init__()


        # Token embedding
        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=1 # ESM TOKENIZER
        )

        # Multi-scale CNN branches
        self.branches = nn.ModuleList([
            ConvBlock(
                in_channels=embed_dim,
                out_channels=conv_dim,
                kernel_size=k
            )
            for k in [3, 5, 7]
        ])

        # Residual projection
        self.res_proj = nn.Conv1d(
            embed_dim,
            conv_dim,
            kernel_size=1,
            bias=False
        )

        # Squeeze-and-Excitation
        self.se = SEBlock(
            channels=conv_dim * 3,
            reduction=4
        )

        self.dropout = nn.Dropout(0.2)

        # Three branches × conv_dim channels
        # Max pooling + mean pooling
        self.hidden_size = conv_dim * 3 * 2

    def forward(self, x, seq_mask):
        """
        Args:
            x:
                [B, L] token IDs

            seq_mask:
                [B, L]
                1 = valid token
                0 = PAD
        Returns:
            CNN feature vector:
                [B, hidden_size]

        For conv_dim=128:
            [B, 768]
        """


        # Validate mask


        if seq_mask is None:
            raise ValueError(
                "seq_mask must be provided to EnhancedCNN1D. "
                "CNN padding masking must never be bypassed."
            )
        # Embedding
        # [B, L] -> [B, L, embed_dim]
        x = self.embed(x)

        # [B, L, embed_dim] -> [B, embed_dim, L]
        x = x.transpose(1, 2)

        # [B, L] -> [B, 1, L]
        mask = seq_mask.unsqueeze(1).to(dtype=x.dtype)

        # Explicitly zero PAD embeddings
        x = x * mask

        # Residual pathway
        res = self.res_proj(x)

        # Remove PAD activations
        res = res * mask

        # Multi-scale branches
        outs = []

        for branch in self.branches:
            y = branch(x, mask)
            # Residual connection
            y = y + res
            # Guarantee PAD = 0 after residual addition
            y = y * mask
            outs.append(y)

        # Concatenate branches
        # [B, 128*3, L]
        combined = torch.cat(
            outs,
            dim=1
        )

        # Guarantee no PAD signal
        combined = combined * mask

        # Mask-aware SE
        combined = self.se(
            combined,
            seq_mask
        )

        # SE can theoretically produce nonzero values
        # at PAD positions, so mask once more.
        combined = combined * mask

        # MASKED GLOBAL MAX POOLING
        # PAD cannot become the maximum.
        combined_for_max = combined.masked_fill(
            seq_mask.unsqueeze(1) == 0,
            torch.finfo(combined.dtype).min
        )

        max_pooled = combined_for_max.max(
            dim=-1
        ).values


        # MASKED GLOBAL MEAN POOLING
        combined_for_mean = combined * mask

        # Actual number of valid tokens
        lengths = seq_mask.sum(
            dim=1,
            keepdim=True
        ).clamp_min(1).to(
            dtype=combined.dtype
        )

        mean_pooled = (
            combined_for_mean.sum(dim=-1)
            / lengths
        )

        # FINAL REPRESENTATION
        pooled = torch.cat(
            [
                max_pooled,
                mean_pooled
            ],
            dim=1
        )

        return self.dropout(pooled)


class MultiHeadAttentionPool(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.attn = nn.Sequential(
            nn.Linear(dim, 128), 
            nn.Tanh(), 
            nn.Linear(128, num_heads)
        )
        self._last_weights = None

    def forward(self, x, mask):
        scores = self.attn(x)
        scores = scores.masked_fill(mask.unsqueeze(-1) == 0, -1e4)
        weights = torch.softmax(scores, dim=1)
        self._last_weights = weights
        pooled = (x.unsqueeze(2) * weights.unsqueeze(-1)).sum(dim=1)
        return pooled.view(x.size(0), -1)


    def orthogonality_loss(self):
        """Penalize overlap between head attention distributions."""
        if self._last_weights is None:
            return 0.0
            
        # weights: [B, L, num_heads] -> transpose to [B, num_heads, L]
        w = self._last_weights.transpose(1, 2)  
        
        # Gram matrix of head attention distributions: shape [B, num_heads, num_heads]
        gram = torch.bmm(w, w.transpose(1, 2))  
        
        # Create a boolean mask for the off-diagonal elements (~torch.eye inverts the identity matrix)
        mask = ~torch.eye(self.num_heads, dtype=torch.bool, device=gram.device)
        
        # Penalize only the off-diagonal overlap to encourage heads to focus on different tokens.
        # We want these dot products to be pushed towards 0.
        loss = (gram[:, mask] ** 2).mean()
        
        return loss
    
class GatedFusion(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(input_dim, input_dim), nn.Sigmoid())
    def forward(self, x):
        return x * self.gate(x)

class PeptideNetwork(nn.Module):
    def __init__(self, num_classes=21, mask_token_id=32):
        super().__init__()
        self.mask_token_id = mask_token_id
        self.num_classes = num_classes

        # BOTH encoders are now the 8M parameter t6 model
        self.esm_t6_a = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=True)        
        self.esm_t6_b = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=False, unfreeze_last_n=2)                                  
        self.cnn = EnhancedCNN1D()          # Bx768

        # REDUCED cross_dim to 128 to save parameters
        cross_dim = 128
        self.proj_t6_a = nn.Linear(320, cross_dim)
        self.proj_t6_b = nn.Linear(320, cross_dim) # Updated to 320 for t6

        self.cross_t6_a = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_a = nn.LayerNorm(cross_dim)
        
        self.cross_t6_b = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_b = nn.LayerNorm(cross_dim)
        
        self.pool_t6_a = MultiHeadAttentionPool(cross_dim, num_heads=4)
        self.pool_t6_b = MultiHeadAttentionPool(cross_dim, num_heads=4)

        # Bottleneck reduction layer before fusion
        # Concat size: 128*4*3 (pools) + 768 (CNN) = 1536 + 768 = 2304
        concat_size = cross_dim * 4 * 2 + self.cnn.hidden_size
        
        self.dim_reduce = nn.Sequential(
            nn.Linear(concat_size, 512),
            nn.GELU()
        )
        
        # Fusion now operates efficiently on 512 dimensions
        self.fusion = GatedFusion(512)
        self.ln = nn.LayerNorm(512)

        # binary head
        binary_features_dim = 64
        self.binary_features = nn.Sequential(
            nn.Linear(512, binary_features_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.binary_classifier = nn.Linear(64, 1) # Final logit

        # Task Query Decoder
        self.task_dim = 128
        self.n_memory_tokens = 16
        self.memory_proj = nn.Sequential(
            nn.Linear(512 + binary_features_dim, self.task_dim * self.n_memory_tokens),
            nn.GELU(),
        )
        self.task_queries = nn.Embedding(num_classes, self.task_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=self.task_dim, nhead=4, dim_feedforward=256,
            batch_first=True, dropout=0.1
        )
        self.task_decoder = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.task_classifiers = nn.ModuleList([
            nn.Linear(self.task_dim, 1) for _ in range(num_classes)
        ])

    def _mask_tokens(self, input_ids, attention_mask, mask_prob=0.15):
        masked_ids = input_ids.clone()
        prob_matrix = torch.full_like(input_ids, mask_prob, dtype=torch.float)
        prob_matrix[attention_mask == 0] = 0
        prob_matrix[:, 0] = 0
        seq_lens = attention_mask.sum(dim=1)
        for i in range(len(seq_lens)):
            if seq_lens[i] > 1:
                prob_matrix[i, seq_lens[i] - 1] = 0
        mask = torch.bernoulli(prob_matrix).bool()
        masked_ids[mask] = self.mask_token_id
        return masked_ids

    def _extract_features(self, seq_input, seq_mask):
        esm6_a_seq = self.esm_t6_a(seq_input, seq_mask)       
        esm6_b_seq = self.esm_t6_b(seq_input, seq_mask)     
        cnn_feat = self.cnn(seq_input, seq_mask)                            

        t6_a = self.proj_t6_a(esm6_a_seq)       
        t6_b = self.proj_t6_b(esm6_b_seq)    


        kv_pad = seq_mask == 0  

        ca_t6_a, _ = self.cross_t6_a(t6_a, t6_b, t6_b, key_padding_mask=kv_pad)
        ca_t6_a = self.ln_ca_t6_a(t6_a + ca_t6_a)

        ca_t6_b, _ = self.cross_t6_b(t6_b, t6_a, t6_a, key_padding_mask=kv_pad)
        ca_t6_b = self.ln_ca_t6_b(t6_b + ca_t6_b)

        pooled_t6_a = self.pool_t6_a(ca_t6_a, seq_mask)
        pooled_t6_b = self.pool_t6_b(ca_t6_b, seq_mask)

        # Concat -> Reduce -> Fuse
        combined = torch.cat([pooled_t6_a, pooled_t6_b, cnn_feat], dim=1)  
        reduced = self.dim_reduce(combined)
        fusion = self.ln(reduced + self.fusion(reduced))

        binary_features = self.binary_features(fusion)
        
        return binary_features, torch.cat([fusion, binary_features], dim=1)
    
    def _binary_classify(self, binary_features):
        
        binary_logits = self.binary_classifier(binary_features)

        return binary_logits

    def _classify(self, final_fusion):
        B = final_fusion.size(0)
        memory = self.memory_proj(final_fusion).view(B, self.n_memory_tokens, self.task_dim)
        tgt = self.task_queries.weight.unsqueeze(0).expand(B, -1, -1)
        decoded = self.task_decoder(tgt, memory)
        logits = torch.cat([self.task_classifiers[i](decoded[:, i, :])
                           for i in range(self.num_classes)], dim=1)
        return logits

    def forward(self, seq_input, seq_mask, mask_tokens=False):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask)
        
        binary_features, combined_features = self._extract_features(seq_input, seq_mask)
        

        return self._binary_classify(binary_features), self._classify(combined_features)


    def ortho_loss(self):
        return (self.pool_t6_a.orthogonality_loss() +
                self.pool_t6_b.orthogonality_loss() 
                ) / 2
    
    def get_features(self, seq_input, seq_mask, mask_tokens=False):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask)
        return self._extract_features(seq_input, seq_mask)

    def multi_classify(self, combined):
        return self._classify(combined)
    
    def binary_classify(self, binary_features):
        return self._binary_classify(binary_features)



# training: function and non-functionl(whole sequence)

In [22]:
import copy
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        # Update parameters (weights & biases)
        for s_param, m_param in zip(self.shadow.parameters(), model.parameters()):
            s_param.data.mul_(self.decay).add_(m_param.data, alpha=1 - self.decay)
        
        # FIX: Update only floating point buffers (skip int buffers like num_batches_tracked)
        for s_buf, m_buf in zip(self.shadow.buffers(), model.buffers()):
            if s_buf.dtype.is_floating_point:
                s_buf.data.mul_(self.decay).add_(m_buf.data, alpha=1 - self.decay)
            else:
                # For integer buffers, just copy directly (no EMA smoothing needed)
                s_buf.data.copy_(m_buf.data)

    def forward(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)

In [23]:
torch.cuda.empty_cache()

model = PeptideNetwork(num_classes=len(endpoints)-1, mask_token_id=32).to(DEVICE)
ema = EMA(model, decay=0.999)

# --- ASL: Asymmetric Loss for Multi-Label Classification ---
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        pos_part = targets * torch.log(p.clamp(min=1e-8))
        neg_p = (1 - p).clamp(min=1e-8)
        if self.clip > 0:
            neg_p = (neg_p + self.clip).clamp(max=1)
        neg_part = (1 - targets) * torch.log(neg_p)
        pos_weight = (1 - p) ** self.gamma_pos
        neg_weight = p ** self.gamma_neg
        loss = -(pos_weight * pos_part + neg_weight * neg_part)
        return loss.mean()

criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.03)
binary_criterion = nn.BCEWithLogitsLoss()

# Optimizer — 3 LR groups (Lowered LRs to prevent early overfitting)
esm_t6_ids = set(id(p) for p in model.esm_t6_a.esm_mlm.parameters())
esm_t6_b_ids = set(id(p) for p in model.esm_t6_b.esm_mlm.parameters())

esm_t6_params = [p for p in model.parameters() if id(p) in esm_t6_ids and p.requires_grad]
esm_t6_b_params = [p for p in model.parameters() if id(p) in esm_t6_b_ids and p.requires_grad]
other_params = [p for p in model.parameters() if id(p) not in esm_t6_ids and id(p) not in esm_t6_b_ids and p.requires_grad]

optimizer = optim.AdamW([
    {'params': esm_t6_params, 'lr': 1e-5},     # Reduced from 1e-5
    {'params': esm_t6_b_params, 'lr': 2e-6},   # Reduced from 2e-6
    {'params': other_params, 'lr': 1e-4},      # Reduced from 1e-4
], weight_decay=0.05)                          # Kept the stronger L2 regularization

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {total_params/1e6:.1f}M | Trainable: {trainable_params/1e6:.1f}M")


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Total: 18.7M | Trainable: 13.6M


In [24]:
NON_FUNC_IDX = endpoint_index['non-functional']
FUNC_INDICES = [i for k, i in endpoint_index.items() if k != 'non-functional']

In [25]:
epochs = 80

In [26]:
# Warmup + Cosine Annealing
steps_per_epoch = len(train_loader)
warmup_epochs = 3
total_steps = epochs * steps_per_epoch

def lr_lambda(step):
    if step < warmup_epochs * steps_per_epoch:
        return step / (warmup_epochs * steps_per_epoch)
    progress = (step - warmup_epochs * steps_per_epoch) / (total_steps - warmup_epochs * steps_per_epoch)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"Warmup ({warmup_epochs} ep) + CosineAnnealing, total {epochs} epochs, {total_steps} steps")


Warmup (3 ep) + CosineAnnealing, total 80 epochs, 225600 steps


In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


In [28]:
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import json

In [ ]:
# --- HELPER FUNCTIONS ---
def calculate_complete_metrics(y_true, y_pred_probs, thresholds, class_names):
    results = {}
    for i in range(y_true.shape[1]):
        y_pred_cls = (y_pred_probs[:, i] >= thresholds[i]).astype(int)
        mcc = matthews_corrcoef(y_true[:, i], y_pred_cls)
        acc = accuracy_score(y_true[:, i], y_pred_cls)
        prec = precision_score(y_true[:, i], y_pred_cls, zero_division=0)
        rec = recall_score(y_true[:, i], y_pred_cls, zero_division=0)
        f1 = f1_score(y_true[:, i], y_pred_cls, zero_division=0)
        try:
            auc_val = roc_auc_score(y_true[:, i], y_pred_probs[:, i])
        except ValueError:
            auc_val = 0.5
        tn, fp, fn, tp = confusion_matrix(y_true[:, i], y_pred_cls, labels=[0, 1]).ravel()
        results[class_names[i]] = {
            "mcc": float(mcc),
            "threshold": float(thresholds[i]),
            "accuracy": float(acc),
            "precision": float(prec),
            "recall": float(rec),
            "f1_score": float(f1),
            "roc_auc": float(auc_val),
            "tp": int(tp),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
        }
    return results

def find_optimal_thresholds(y_true, y_pred_probs):
    optimal_thresholds = []
    for i in range(y_true.shape[1]):
        best_mcc = -1
        best_threshold = 0.5
        for threshold in np.arange(0.1, 0.95, 0.05):
            y_pred_class = (y_pred_probs[:, i] >= threshold).astype(int)
            mcc = matthews_corrcoef(y_true[:, i], y_pred_class)
            if mcc > best_mcc:
                best_mcc = mcc
                best_threshold = threshold
        optimal_thresholds.append(best_threshold)
    return optimal_thresholds

def calculate_mcc(tp, tn, fp, fn):
    num = (tp * tn) - (fp * fn)
    den_sq = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    return num / math.sqrt(den_sq) if den_sq > 0 else 0.0

def aggregate_mcc(*metric_groups):
    tp, tn, fp, fn = 0, 0, 0, 0
    for metrics in metric_groups:
        for values in metrics.values():
            tp += values['tp']
            tn += values['tn']
            fp += values['fp']
            fn += values['fn']
    return calculate_mcc(tp, tn, fp, fn)

def feature_mixup(features, labels, alpha=0.4):
    batch_size = features.size(0)
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(batch_size, device=features.device)
    mixed_features = lam * features + (1 - lam) * features[index]
    mixed_labels = lam * labels + (1 - lam) * labels[index]
    return mixed_features, mixed_labels

def rdrop_kl_loss(logits1, logits2):
    kl1 = F.binary_cross_entropy_with_logits(logits1, torch.sigmoid(logits2).detach(), reduction='mean')
    kl2 = F.binary_cross_entropy_with_logits(logits2, torch.sigmoid(logits1).detach(), reduction='mean')
    return (kl1 + kl2) / 2

def enable_dropout(model):
    """
    Specifically tailored for PeptideNetwork to activate all sources of dropout
    during Test-Time Augmentation (TTA), including functional dropouts hidden 
    inside complex PyTorch modules.
    """
    for module in model.modules():
        class_name = module.__class__.__name__
        
        # 1. Standard explicit dropout layers (Dropout, Dropout1d, Dropout2d)
        if class_name.startswith('Dropout'):
            module.train()
            
        # 2. Cross-Attention functional dropout
        elif class_name == 'MultiheadAttention':
            module.train()
            
        # 3. Task Query Decoder functional dropout
        elif class_name in ['TransformerDecoder', 'TransformerDecoderLayer']:
            module.train()
            
        # 4. GRU functional dropout (applied between internal layers)
        elif class_name == 'GRU':
            module.train()


# --- SWA ---
swa_start_epoch = 30
swa_model = None
swa_n = 0

def update_swa(model_state, swa_state, n):
    if swa_state is None:
        return {k: v.clone() for k, v in model_state.items()}, 1
    for k in swa_state:
        swa_state[k] = (swa_state[k] * n + model_state[k]) / (n + 1)
    return swa_state, n + 1

# --- MAIN TRAINING LOOP ---
history_log = []
history_log_path = "phase_1_model.json"
best_model_path = "phase_1_model.pt"
best_val_mcc = -1.0
scaler = torch.amp.GradScaler('cuda')
patience_counter = 0
early_stop_patience = 20
mixup_alpha = 0.4
rdrop_alpha = 1.0
tta_passes = 5

func_names = [index_endpoint[i] for i in FUNC_INDICES]
bin_names = ["NON-FUNCTIONAL"]
bin_thresh = [0.5]
spec_thresh = [0.5] * len(func_names)

print(f"  R-Drop α={rdrop_alpha} | Mixup α={mixup_alpha} | SWA from ep {swa_start_epoch}")
print(f"  Validation eval: TTA={tta_passes} | Test eval: TTA={tta_passes}")
print("=" * 70)
for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0

    for data in train_loader:
        input_ids = data['input_ids'].to(DEVICE)
        attention_mask = data['attention_mask'].to(DEVICE)
        labels = data['labels'].to(DEVICE)

        optimizer.zero_grad()

        # Binary label: 1 if NON-FUNCTIONAL, 0 if FUNCTIONAL
        target_bin = labels[:, NON_FUNC_IDX].unsqueeze(1).float()
        target_spec = labels[:, FUNC_INDICES]

        with torch.amp.autocast('cuda'):
            # Two augmented passes for R-Drop
            bin_features1, features1 = model.get_features(input_ids, attention_mask, mask_tokens=True)
            bin_features2, features2 = model.get_features(input_ids, attention_mask, mask_tokens=True)

            # 1. Clean predictions
            logits1_clean = model.multi_classify(features1)
            logits2_clean = model.multi_classify(features2)

            # 2. Binary classification (Always on the whole batch)
            binary_logits = model.binary_classify(bin_features2)
            binary_loss = binary_criterion(binary_logits, target_bin)
            ortho = model.ortho_loss()

            # --- THE FIX: Boolean Masking ---
            is_functional_mask = (target_bin == 0).view(-1).bool()
            
            if is_functional_mask.sum() > 0:
                # Slice out ONLY the functional peptides
                func_f1 = features1[is_functional_mask]
                func_targets = target_spec[is_functional_mask]
                func_logits1 = logits1_clean[is_functional_mask]
                func_logits2 = logits2_clean[is_functional_mask]

                # R-Drop only on functional
                rdrop_loss = rdrop_kl_loss(func_logits1, func_logits2)
                
                # Mixup only on functional
                mixed_f, mixed_y = feature_mixup(func_f1, func_targets, alpha=mixup_alpha)
                logits_mix = model.multi_classify(mixed_f)
                
                # ASL evaluates ONLY the functional slice
                asl_loss_mix = criterion(logits_mix, mixed_y)
                asl_loss_clean = criterion(func_logits2, func_targets)
                
                spec_loss = (asl_loss_mix + asl_loss_clean) / 2 + (rdrop_alpha * rdrop_loss) + (0.1 * ortho)
                loss = 0.3 * binary_loss + 0.7 * spec_loss
            else:
                # If batch is 100% non-functional
                loss = binary_loss + (0.1 * ortho)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema.update(model)

        total_train_loss += loss.item() * input_ids.size(0)

    # --- SWA Update ---
    if epoch >= swa_start_epoch:
        swa_model, swa_n = update_swa(ema.shadow.state_dict(), swa_model, swa_n)

    if swa_model is not None and epoch >= swa_start_epoch:
        eval_model = copy.deepcopy(ema.shadow)
        eval_model.load_state_dict(swa_model)
        eval_model.eval()
    else:
        eval_model = ema.shadow
        eval_model.eval()

    # --- VALIDATION (TTA) ---
    total_val_loss = 0.0
    val_bin_y = []
    val_spec_y = []
    val_tta_bin_preds = [[] for _ in range(tta_passes)]
    val_tta_spec_preds = [[] for _ in range(tta_passes)]

    with torch.no_grad():
        for data in val_loader:
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            labels = data['labels'].to(DEVICE)

            target_bin = labels[:, NON_FUNC_IDX].unsqueeze(1).float()
            target_spec = labels[:, FUNC_INDICES]

            with torch.amp.autocast('cuda'):
                pred_bin, pred_spec = eval_model(input_ids, attention_mask)
                loss_bin = binary_criterion(pred_bin, target_bin)

                is_functional_mask = (target_bin == 0).view(-1).bool()
                if is_functional_mask.sum() > 0:
                    loss_spec = criterion(pred_spec[is_functional_mask], target_spec[is_functional_mask])
                    loss_val = 0.3 * loss_bin + 0.7 * loss_spec
                else:
                    loss_val = loss_bin  # fallback if batch has no functional peptides

            total_val_loss += loss_val.item() * input_ids.size(0)
            
            # Append entire batch for both binary and specific targets
            val_bin_y.append(target_bin.cpu().numpy())
            val_spec_y.append(target_spec.cpu().numpy()) 
            
            val_tta_bin_preds[0].append(torch.sigmoid(pred_bin).detach().cpu().numpy())
            val_tta_spec_preds[0].append(torch.sigmoid(pred_spec).detach().cpu().numpy())

            if tta_passes > 1:
                enable_dropout(eval_model)
                for t in range(1, tta_passes):
                    with torch.amp.autocast('cuda'):
                        pred_bin_t, pred_spec_t = eval_model(input_ids, attention_mask)
                    val_tta_bin_preds[t].append(torch.sigmoid(pred_bin_t).detach().cpu().numpy())
                    val_tta_spec_preds[t].append(torch.sigmoid(pred_spec_t).detach().cpu().numpy())
                eval_model.eval()

    y_true_bin_val = np.concatenate(val_bin_y)
    y_pred_bin_val = np.mean([np.concatenate(pred_list) for pred_list in val_tta_bin_preds], axis=0)
    
    y_true_spec_val = np.concatenate(val_spec_y)
    raw_pred_spec_val = np.mean([np.concatenate(pred_list) for pred_list in val_tta_spec_preds], axis=0)

    # SOFT GATING: Scale specific predictions by functionality probability
    # y_pred_bin_val is probability of NON-FUNCTIONAL, so 1 - y_pred_bin_val is prob of FUNCTIONAL
    y_pred_spec_val = raw_pred_spec_val * (1 - y_pred_bin_val)

    # Compute optimal thresholds on Validation set
    opt_bin_thresh = find_optimal_thresholds(y_true_bin_val.reshape(-1, 1), y_pred_bin_val.reshape(-1, 1))
    opt_spec_thresh = find_optimal_thresholds(y_true_spec_val, raw_pred_spec_val)


    val_bin_metrics = calculate_complete_metrics(y_true_bin_val, y_pred_bin_val, opt_bin_thresh, bin_names)
    val_spec_metrics = calculate_complete_metrics(y_true_spec_val, y_pred_spec_val, opt_spec_thresh, func_names)
    val_total_mcc = aggregate_mcc(val_bin_metrics, val_spec_metrics)
    
    val_func_mccs = [metrics['mcc'] for metrics in val_spec_metrics.values() if not np.isnan(metrics['mcc'])]
    val_avg_func_mcc = sum(val_func_mccs) / len(val_func_mccs) if val_func_mccs else 0.0


    # --- TEST (TTA) ---
    total_test_loss = 0.0
    test_bin_y = []
    test_spec_y = []
    test_tta_bin_preds = [[] for _ in range(tta_passes)]
    test_tta_spec_preds = [[] for _ in range(tta_passes)]

    with torch.no_grad():
        for data in test_loader:
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            labels = data['labels'].to(DEVICE)

            target_bin = labels[:, NON_FUNC_IDX].unsqueeze(1).float()
            target_spec = labels[:, FUNC_INDICES]

            with torch.amp.autocast('cuda'):
                pred_bin, pred_spec = eval_model(input_ids, attention_mask)
                loss_bin = binary_criterion(pred_bin, target_bin)
                
                is_functional_mask = (target_bin == 0).view(-1).bool()
                if is_functional_mask.sum() > 0:
                    loss_spec = criterion(pred_spec[is_functional_mask], target_spec[is_functional_mask])
                    loss_test = 0.3 * loss_bin + 0.7 * loss_spec
                else:
                    loss_test = loss_bin

            total_test_loss += loss_test.item() * input_ids.size(0)
            
            test_bin_y.append(target_bin.cpu().numpy())
            test_spec_y.append(target_spec.cpu().numpy())
            
            # First pass predictions
            test_tta_bin_preds[0].append(torch.sigmoid(pred_bin).detach().cpu().numpy())
            test_tta_spec_preds[0].append(torch.sigmoid(pred_spec).detach().cpu().numpy())

            # Additional TTA passes
            if tta_passes > 1:
                enable_dropout(eval_model)
                for t in range(1, tta_passes):
                    with torch.amp.autocast('cuda'):
                        pred_bin_t, pred_spec_t = eval_model(input_ids, attention_mask)
                    test_tta_bin_preds[t].append(torch.sigmoid(pred_bin_t).detach().cpu().numpy())
                    test_tta_spec_preds[t].append(torch.sigmoid(pred_spec_t).detach().cpu().numpy())
                eval_model.eval()

    y_true_bin_test = np.concatenate(test_bin_y)
    y_pred_bin_test = np.mean([np.concatenate(pred_list) for pred_list in test_tta_bin_preds], axis=0)
    
    y_true_spec_test = np.concatenate(test_spec_y)
    raw_pred_spec_test = np.mean([np.concatenate(pred_list) for pred_list in test_tta_spec_preds], axis=0)

    # SOFT GATING for Test
    y_pred_spec_test = raw_pred_spec_test * (1 - y_pred_bin_test)

    test_bin_metrics = calculate_complete_metrics(y_true_bin_test, y_pred_bin_test, opt_bin_thresh, bin_names)
    test_spec_metrics = calculate_complete_metrics(y_true_spec_test, y_pred_spec_test, opt_spec_thresh, func_names)
    test_total_mcc = aggregate_mcc(test_bin_metrics, test_spec_metrics)

    test_func_mccs = [metrics['mcc'] for metrics in test_spec_metrics.values() if not np.isnan(metrics['mcc'])]
    test_avg_func_mcc = sum(test_func_mccs) / len(test_func_mccs) if test_func_mccs else 0.0

    avg_train_loss = total_train_loss / len(train_loader.dataset)
    avg_val_loss = total_val_loss / len(val_loader.dataset)
    avg_test_loss = total_test_loss / len(test_loader.dataset)

    if val_avg_func_mcc > best_val_mcc:
        best_val_mcc = val_avg_func_mcc
        patience_counter = 0
        torch.save(eval_model.state_dict(), best_model_path)
        marker = " >>> BEST <<<"
    else:
        patience_counter += 1
        marker = ""

    lr_now = optimizer.param_groups[0]['lr']
    swa_tag = f" [SWA n={swa_n}]" if epoch >= swa_start_epoch else ""
    print(f"Ep {epoch:02d}/{epochs} | LR: {lr_now:.1e} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | Test: {avg_test_loss:.4f}{swa_tag}")
    print(f"  MCC -> Func Avg (Val): {val_avg_func_mcc:.4f} | Func Avg (Test): {test_avg_func_mcc:.4f} | Pat: {patience_counter}/{early_stop_patience}{marker}")
    print(f"  Overall Total MCC -> Val: {val_total_mcc:.4f} | Test: {test_total_mcc:.4f}")
    print(f"  NON-FUNCTIONAL MCC -> Val: {val_bin_metrics['NON-FUNCTIONAL']['mcc']:.4f} | Test: {test_bin_metrics['NON-FUNCTIONAL']['mcc']:.4f}")
    
    for name in func_names:
        val_mcc = val_spec_metrics[name]['mcc'] if name in val_spec_metrics else float('nan')
        test_mcc = test_spec_metrics[name]['mcc'] if name in test_spec_metrics else float('nan')
        print(f"  {name:16s}: Val MCC={val_mcc:.4f} | Test MCC={test_mcc:.4f}")
    print("-" * 70)

    epoch_data = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "test_loss": avg_test_loss,
        "val_avg_func_mcc": val_avg_func_mcc,
        "val_total_mcc": float(val_total_mcc),
        "test_total_mcc": float(test_total_mcc),
        "test_avg_func_mcc": float(test_avg_func_mcc),
        "thresholds": {
            "binary": opt_bin_thresh,
            "functional": opt_spec_thresh,
        },
        "val_metrics": {**val_bin_metrics, **val_spec_metrics},
        "test_metrics": {**test_bin_metrics, **test_spec_metrics},
    }
    history_log.append(epoch_data)
    with open(history_log_path, "w") as f:
        json.dump(history_log, f, indent=2)

    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping at epoch {epoch}!")
        break

    if swa_model is not None and epoch >= swa_start_epoch:
        del eval_model

print(f"\nTraining Complete! Best Val MCC: {best_val_mcc:.4f}")

  R-Drop α=1.0 | Mixup α=0.4 | SWA from ep 30
  Validation eval: TTA=5 | Test eval: TTA=5
Ep 01/80 | LR: 3.3e-06 | Train: 0.5661 | Val: 0.0709 | Test: 0.0708
  MCC -> Func Avg (Val): 0.2467 | Func Avg (Test): 0.2457 | Pat: 0/20 >>> BEST <<<
  Overall Total MCC -> Val: 0.6825 | Test: 0.6732
  NON-FUNCTIONAL MCC -> Val: 0.9193 | Test: 0.9163
  anti-bacterial  : Val MCC=0.6618 | Test MCC=0.6840
  anti-cancer     : Val MCC=0.2478 | Test MCC=0.2406
  anti-fungal     : Val MCC=0.1614 | Test MCC=0.1337
  anti-parasitic  : Val MCC=0.1479 | Test MCC=0.1830
  anti-viral      : Val MCC=0.1610 | Test MCC=0.2106
  cell-cell-communication: Val MCC=0.0000 | Test MCC=0.0000
  drug-delivery   : Val MCC=0.1598 | Test MCC=0.0971
  immunological   : Val MCC=0.2686 | Test MCC=0.2655
  inhibitor       : Val MCC=0.0588 | Test MCC=0.0944
  metabolic       : Val MCC=0.0140 | Test MCC=0.0177
  other-functional: Val MCC=0.1282 | Test MCC=0.1454
  signal-peptide  : Val MCC=0.6639 | Test MCC=0.6447
  toxic        

# creating non-functional subsequence dataset

In [78]:
# reading dataset
train_df = pd.read_csv('train_classification.csv', index_col=0)
val_df = pd.read_csv('validation_classification.csv', index_col=0)
test_df = pd.read_csv('test_classification.csv', index_col=0)

In [79]:
# getting functional peptide distributions
combined_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

functional_lens = []
# taking lengths of function sequences
for i, row in combined_df.iterrows():
    if row['non-functional'] is False:
        functional_lens.append(len(row['sequence']))

unique_lengths, counts = np.unique(functional_lens, return_counts=True)
probs = counts / counts.sum()
func_length_dist = {
    'lengths': unique_lengths.tolist(),
    'probs': probs.tolist()
}

In [80]:
def get_non_functional_subsequences(non_functional_sequences, functional_len_distribution):
    def _sample_length_from_func_dist(functional_len_distribution):
        """Sample a length from the functional peptide length distribution."""
        return np.random.choice(
            functional_len_distribution['lengths'],
            p=functional_len_distribution['probs']
        )
    def _get_substring(seq, target_length):
        """Extract a random continuous substring of target_length from seq."""
        seq_len = len(seq)
        if seq_len <= target_length:
            return seq
        start = random.randint(0, seq_len - target_length)
        return seq[start:start + target_length]

    non_func_substrings = []
    for seq in non_functional_sequences:
        target_len = _sample_length_from_func_dist(functional_len_distribution)
        non_func_substrings.append(_get_substring(seq, target_len))

    return non_func_substrings



In [81]:
len(train_df[train_df['non-functional']]), len(train_df)

(113793, 180471)

In [82]:
train_non_func = train_df[train_df['non-functional']]['sequence'].tolist()
val_non_func = val_df[val_df['non-functional']]['sequence'].tolist()
test_non_func = test_df[test_df['non-functional']]['sequence'].tolist()

train_non_func_subseq_set = set()
val_non_func_subseq_set = set()
test_non_func_subseq_set = set()
for _ in range(10):
    sub_seqs = get_non_functional_subsequences(train_non_func, func_length_dist)
    train_non_func_subseq_set = train_non_func_subseq_set.union(set(sub_seqs))

    sub_seqs = get_non_functional_subsequences(val_non_func, func_length_dist)
    val_non_func_subseq_set = val_non_func_subseq_set.union(set(sub_seqs))

    sub_seqs = get_non_functional_subsequences(test_non_func, func_length_dist)
    test_non_func_subseq_set = test_non_func_subseq_set.union(set(sub_seqs))
    

In [83]:
len(train_non_func_subseq_set), len(val_non_func_subseq_set), len(test_non_func_subseq_set)

(1119519, 162784, 322618)

In [84]:
train_non_func_subseq_set = train_non_func_subseq_set - set(train_df['sequence'].tolist())
val_non_func_subseq_set = val_non_func_subseq_set - set(val_df['sequence'].tolist())
test_non_func_subseq_set = test_non_func_subseq_set - set(test_df['sequence'].tolist())


In [85]:
len(train_non_func_subseq_set), len(val_non_func_subseq_set), len(test_non_func_subseq_set)

(1098907, 159696, 316685)

# mmseqs 

In [86]:
import os
import subprocess
import tempfile
from pathlib import Path

In [87]:
def write_mmseqs_fasta(sequences, fasta_file, prefix="seq"):
    """
    Write a Python iterable/set of peptide sequences to FASTA.
    """
    sequences = list(sequences)

    with open(fasta_file, "w") as f:
        for i, seq in enumerate(sequences):
            seq = str(seq).strip().upper()

            if not seq:
                continue

            f.write(f">{prefix}_{i}\n")
            f.write(f"{seq}\n")

    return len(sequences)


def run_mmseqs_remove_similar(
    query_sequences,
    functional_sequences,
    output_prefix,
    min_seq_id=0.70,
    min_coverage=0.80,
    threads=8,
    tmp_dir=None,
):
    """
    Remove query sequences that have strong sequence similarity
    to any functional reference sequence.

    Similarity criterion:
        sequence identity >= min_seq_id
        AND
        query coverage >= min_coverage

    Returns:
        filtered_sequences: set of sequences that survived
        similar_sequences: set of sequences removed
    """

    query_sequences = set(query_sequences)
    functional_sequences = set(functional_sequences)

    if len(query_sequences) == 0:
        return set(), set()

    if len(functional_sequences) == 0:
        print("[MMseqs2] Functional reference database is empty.")
        return query_sequences, set()

    output_prefix = Path(output_prefix)
    output_prefix.parent.mkdir(parents=True, exist_ok=True)

    work_dir = output_prefix.parent / f"{output_prefix.name}_work"

    # Remove previous MMseqs2 intermediate files from an earlier run
    if work_dir.exists():
        import shutil
        shutil.rmtree(work_dir)

    work_dir.mkdir(parents=True, exist_ok=True)

    query_fasta = work_dir / "queries.fasta"
    target_fasta = work_dir / "functional.fasta"

    query_db = work_dir / "queryDB"
    target_db = work_dir / "targetDB"
    result_db = work_dir / "resultDB"

    tmp = (
        Path(tmp_dir)
        if tmp_dir is not None
        else work_dir / "tmp"
    )
    tmp.mkdir(parents=True, exist_ok=True)

    # ---------------------------------------------------------
    # 1. Write FASTA files
    # ---------------------------------------------------------

    print(f"[MMseqs2] Queries: {len(query_sequences):,}")
    print(f"[MMseqs2] Functional reference: {len(functional_sequences):,}")

    write_mmseqs_fasta(
        query_sequences,
        query_fasta,
        prefix="query"
    )

    write_mmseqs_fasta(
        functional_sequences,
        target_fasta,
        prefix="functional"
    )

    # ---------------------------------------------------------
    # 2. Create MMseqs2 databases
    # ---------------------------------------------------------

    subprocess.run(
        [
            "mmseqs",
            "createdb",
            str(query_fasta),
            str(query_db)
        ],
        check=True
    )

    subprocess.run(
        [
            "mmseqs",
            "createdb",
            str(target_fasta),
            str(target_db)
        ],
        check=True
    )

    # ---------------------------------------------------------
    # 3. Similarity search
    # ---------------------------------------------------------
    #
    # We use:
    #
    #   --min-seq-id       minimum sequence identity
    #   -c                 query coverage
    #   --cov-mode 0       coverage relative to query
    #
    # This is important because your query is the generated
    # pseudo-negative peptide.
    #
    # Example:
    #
    # 70% identity AND 80% of query covered
    #
    # ---------------------------------------------------------

    subprocess.run(
        [
            "mmseqs",
            "search",
            str(query_db),
            str(target_db),
            str(result_db),
            str(tmp),

            "-s", "7.5",

            "--min-seq-id", str(min_seq_id),

            "-c", str(min_coverage),

            "--cov-mode", "0",

            "--threads", str(threads),

            "-e", "1e-3",

            "--max-seqs", "20",
        ],
        check=True
    )

    # ---------------------------------------------------------
    # 4. Convert result DB to TSV
    # ---------------------------------------------------------

    result_tsv = work_dir / "similarity.tsv"

    subprocess.run(
        [
            "mmseqs",
            "convertalis",

            str(query_db),
            str(target_db),
            str(result_db),
            str(result_tsv),

            "--format-output",
            "query,target,pident,alnlen,qcov,tcov,evalue,bits"
        ],
        check=True
    )

    # ---------------------------------------------------------
    # 5. Identify queries having functional-like matches
    # ---------------------------------------------------------

    similar_ids = set()

    if result_tsv.exists() and result_tsv.stat().st_size > 0:

        with open(result_tsv, "r") as f:

            for line in f:

                fields = line.rstrip("\n").split("\t")

                if len(fields) < 8:
                    continue

                query_id = fields[0]

                pident = float(fields[2])
                qcov = float(fields[4])

                # Explicit safety check.
                #
                # MMseqs2 already filtered using these thresholds,
                # but we check again during parsing.
                if (
                    pident >= min_seq_id * 100.0
                    and qcov >= min_coverage
                ):
                    similar_ids.add(query_id)

    # ---------------------------------------------------------
    # 6. Map query IDs back to sequences
    # ---------------------------------------------------------

    query_list = list(query_sequences)

    removed_sequences = set()

    for i, seq in enumerate(query_list):

        query_id = f"query_{i}"

        if query_id in similar_ids:
            removed_sequences.add(seq)

    filtered_sequences = query_sequences - removed_sequences

    print()
    print("========== MMseqs2 similarity filtering ==========")
    print(f"Input sequences       : {len(query_sequences):,}")
    print(f"Similar to functional : {len(removed_sequences):,}")
    print(f"Remaining              : {len(filtered_sequences):,}")
    print(
        f"Removed                 : "
        f"{100 * len(removed_sequences) / len(query_sequences):.2f}%"
    )
    print("===================================================")

    return filtered_sequences, removed_sequences


def mmseqs_cluster_representatives(
    sequences,
    output_prefix,
    min_seq_id=0.70,
    coverage=0.80,
    threads=8,
):
    """
    Cluster sequences and return one representative per cluster.
    """

    sequences = set(sequences)

    if len(sequences) == 0:
        return set()

    output_prefix = Path(output_prefix)
    output_prefix.parent.mkdir(parents=True, exist_ok=True)

    work_dir = output_prefix.parent / f"{output_prefix.name}_work"
    work_dir.mkdir(parents=True, exist_ok=True)

    fasta = work_dir / "input.fasta"
    db = work_dir / "inputDB"
    cluster_db = work_dir / "clusterDB"
    tmp = work_dir / "tmp"

    tmp.mkdir(parents=True, exist_ok=True)

    write_mmseqs_fasta(
        sequences,
        fasta,
        prefix="seq"
    )

    subprocess.run(
        [
            "mmseqs",
            "createdb",
            str(fasta),
            str(db)
        ],
        check=True
    )

    subprocess.run(
        [
            "mmseqs",
            "cluster",
            str(db),
            str(cluster_db),
            str(tmp),

            "--min-seq-id", str(min_seq_id),

            "-c", str(coverage),

            "--cov-mode", "0",

            "-s", "7.5",

            "--threads", str(threads),
        ],
        check=True
    )

    representatives_fasta = work_dir / "representatives.fasta"

    subprocess.run(
        [
            "mmseqs",
            "result2repseq",
            str(db),
            str(cluster_db),
            str(cluster_db) + "_rep"
        ],
        check=True
    )

    subprocess.run(
        [
            "mmseqs",
            "convert2fasta",
            str(cluster_db) + "_rep",
            str(representatives_fasta)
        ],
        check=True
    )

    # Parse representative FASTA
    representatives = set()

    with open(representatives_fasta, "r") as f:

        sequence = []

        for line in f:

            line = line.strip()

            if line.startswith(">"):

                if sequence:
                    representatives.add(
                        "".join(sequence)
                    )

                sequence = []

            else:
                sequence.append(line)

        if sequence:
            representatives.add(
                "".join(sequence)
            )

    print()
    print("========== MMseqs2 clustering ==========")
    print(f"Input       : {len(sequences):,}")
    print(f"Representatives: {len(representatives):,}")
    print(
        f"Reduction   : "
        f"{100 * (1 - len(representatives) / len(sequences)):.2f}%"
    )
    print("=========================================")

    return representatives

In [88]:
# ============================================================
# PHASE 2: FUNCTIONAL SIMILARITY FILTER
# ============================================================

train_non_func_subseq_set, _ = run_mmseqs_remove_similar(
    train_non_func_subseq_set,
    train_df[train_df['non-functional'] == False]['sequence'],
    "./mmseqs_phase2/train_functional_filter",
    min_seq_id=0.70,
    min_coverage=0.80,
    threads=8
)

val_non_func_subseq_set, _ = run_mmseqs_remove_similar(
    val_non_func_subseq_set,
    train_df[train_df['non-functional'] == False]['sequence'],
    "./mmseqs_phase2/val_functional_filter",
    min_seq_id=0.70,
    min_coverage=0.80,
    threads=8
)

test_non_func_subseq_set, _ = run_mmseqs_remove_similar(
    test_non_func_subseq_set,
    train_df[train_df['non-functional'] == False]['sequence'],
    "./mmseqs_phase2/test_functional_filter",
    min_seq_id=0.70,
    min_coverage=0.80,
    threads=8
)


[MMseqs2] Queries: 1,098,907
[MMseqs2] Functional reference: 66,678
createdb mmseqs_phase2/train_functional_filter_work/queries.fasta mmseqs_phase2/train_functional_filter_work/queryDB 

MMseqs Version:       	13.45111
Database type         	0
Shuffle input database	true
Createdb mode         	0
Write lookup file     	1
Offset of numeric ids 	0
Compressed            	0
Verbosity             	3

Converting sequences
[===================================================================================================	1 Mio. sequences processed
Time for merging to queryDB_h: 0h 0m 0s 125ms
Time for merging to queryDB: 0h 0m 0s 142ms
Database type: Aminoacid
Time for processing: 0h 0m 2s 438ms
createdb mmseqs_phase2/train_functional_filter_work/functional.fasta mmseqs_phase2/train_functional_filter_work/targetDB 

MMseqs Version:       	13.45111
Database type         	0
Shuffle input database	true
Createdb mode         	0
Write lookup file     	1
Offset of numeric ids 	0
Compressed         

## giving non-functional sub-seq train, val and test to inference, and only considering a seq as true non-func if multihead is predicting it as non-func

In [ ]:
#### start from here

In [94]:
# loading classification model
MODEL_PATH = 'phase_1_model.pt'
THRESHOLD_PATH = 'phase_1_model.json'
DEVICE = 'cuda:0'

model = PeptideNetwork(num_classes=13, mask_token_id=32)
if os.path.exists(MODEL_PATH):
    print(f"Found {MODEL_PATH} ! Loading weights.")
    state_dict = torch.load(MODEL_PATH, map_location="cpu")
else:
    raise("Wrong model path")

esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE)

func_names = [index_endpoint[i] for i in FUNC_INDICES]
bin_names = ["NON-FUNCTIONAL"]

# reading threasholds 
with open(THRESHOLD_PATH) as f:
    data = json.load(f)
    best_i = 0
    for i, e in enumerate(data):
        if e['val_avg_func_mcc'] > data[best_i]['val_avg_func_mcc']:
            best_i = i

    print(f'reading thresholds from {best_i}th epoch, as it gave the best val mcc')
    threshold_binary = data[best_i]['thresholds']['binary']
    threshold_functional = data[best_i]['thresholds']['functional']

def enable_dropout(model):
    """
    Specifically tailored for PeptideNetwork to activate all sources of dropout
    during Test-Time Augmentation (TTA), including functional dropouts hidden 
    inside complex PyTorch modules.
    """
    for module in model.modules():
        class_name = module.__class__.__name__
        
        # 1. Standard explicit dropout layers (Dropout, Dropout1d, Dropout2d)
        if class_name.startswith('Dropout'):
            module.train()
            
        # 2. Cross-Attention functional dropout
        elif class_name == 'MultiheadAttention':
            module.train()
            
        # 3. Task Query Decoder functional dropout
        elif class_name in ['TransformerDecoder', 'TransformerDecoderLayer']:
            module.train()
            
        # 4. GRU functional dropout (applied between internal layers)
        elif class_name == 'GRU':
            module.train()

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Found phase_1_model.pt ! Loading weights.
reading thresholds from 76th epoch, as it gave the best val mcc


In [95]:
BATCH_SIZE = 256
TTA_PASSES = 5


In [96]:
def inference(model, sequences):
    test_tta_bin_preds = [[] for _ in range(TTA_PASSES)]
    test_tta_spec_preds = [[] for _ in range(TTA_PASSES)]

    with torch.no_grad():
        model.eval()
        for i in range(0, len(sequences), BATCH_SIZE):
            batch = sequences[i: i+BATCH_SIZE]
            
            # FIX: Tokenize the 'batch', not the entire 'sequences' list!
            encodings = esm_tokenizer(batch, add_special_tokens=True, max_length=100,
                                        padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt')

            input_ids = encodings['input_ids'].to(DEVICE)
            attention_mask = encodings['attention_mask'].to(DEVICE)


            pred_bin, pred_spec = model(input_ids, attention_mask)

            # First pass predictions
            test_tta_bin_preds[0].append(torch.sigmoid(pred_bin).detach().cpu().numpy())
            test_tta_spec_preds[0].append(torch.sigmoid(pred_spec).detach().cpu().numpy())

            # Additional TTA passes
            if TTA_PASSES > 1:
                enable_dropout(model)
                for t in range(1, TTA_PASSES):
                    pred_bin_t, pred_spec_t = model(input_ids, attention_mask)
                    test_tta_bin_preds[t].append(torch.sigmoid(pred_bin_t).detach().cpu().numpy())
                    test_tta_spec_preds[t].append(torch.sigmoid(pred_spec_t).detach().cpu().numpy())
                model.eval()

        y_pred_bin_test = np.mean([np.concatenate(pred_list) for pred_list in test_tta_bin_preds], axis=0)

        raw_pred_spec_test = np.mean([np.concatenate(pred_list) for pred_list in test_tta_spec_preds], axis=0)

        # SOFT GATING for Test
        y_pred_spec_test = raw_pred_spec_test * (1 - y_pred_bin_test)

        return y_pred_bin_test, y_pred_spec_test

def get_predicted_non_functional_sequences(model, sequences, threshold_binary, threshold_functional):
    sequences = list(sequences)
    bin_preds, multi_preds =  inference(model, sequences)

    non_func_seqs = set()
    for i in range(len(sequences)):
        
        # Check binary (NON-FUNCTIONAL) prediction
        prob_bin = float(bin_preds[i][0])
        if prob_bin >= threshold_binary[0]: 
            non_func_seqs.add(sequences[i])
            continue

        # Check functional predictions
        non_fun = True
        for j in range(len(multi_preds[i])):
            prob_spec = float(multi_preds[i][j])
            if prob_spec >= threshold_functional[j]:
                non_fun = False

        if non_fun:
            non_func_seqs.add(sequences[i])
    
    return non_func_seqs

In [97]:
train_non_func_predict_subseqs = get_predicted_non_functional_sequences(model,
    train_non_func_subseq_set,threshold_binary, threshold_functional)


In [100]:
print("Train->","sub seq count:", len(train_non_func_subseq_set), "predict sub seq count:", len(train_non_func_predict_subseqs))

Train-> sub seq count: 1094354 predict sub seq count: 526873


In [99]:
val_non_func_predict_subseqs = get_predicted_non_functional_sequences(model,
    val_non_func_subseq_set,threshold_binary, threshold_functional)

In [101]:
print("Val->","sub seq count:", len(val_non_func_subseq_set), "predict sub seq count:", len(val_non_func_predict_subseqs))

Val-> sub seq count: 159213 predict sub seq count: 75816


In [102]:
test_non_func_predict_subseqs = get_predicted_non_functional_sequences(model,
    test_non_func_subseq_set,threshold_binary, threshold_functional)


In [103]:
print("Val->","sub seq count:", len(test_non_func_subseq_set), "predict sub seq count:", len(test_non_func_predict_subseqs))

Val-> sub seq count: 315746 predict sub seq count: 151614


In [104]:
# saving
with open("train_non_functional_subseq.txt", 'w') as f:
    for seq in train_non_func_predict_subseqs:
        f.write(seq + '\n')

with open("val_non_functional_subseq.txt", 'w') as f:
    for seq in val_non_func_predict_subseqs:
        f.write(seq + '\n')

with open("test_non_functional_subseq.txt", 'w') as f:
    for seq in test_non_func_predict_subseqs:
        f.write(seq + '\n')

In [105]:
# cdhit
# reading dataset
train_df = pd.read_csv('train_classification.csv', index_col=0)
val_df = pd.read_csv('validation_classification.csv', index_col=0)
test_df = pd.read_csv('test_classification.csv', index_col=0)


In [106]:
cd_hit_map = dict()

for seq in train_df[train_df['non-functional'] == False]['sequence'].tolist():
    i = len(cd_hit_map)
    cd_hit_map[i] = [seq, 'train_fun']

#####
for seq in val_df[val_df['non-functional'] == False]['sequence'].tolist():
    i = len(cd_hit_map)
    cd_hit_map[i] = [seq, 'val_fun']

#####

for seq in test_df[test_df['non-functional'] == False]['sequence'].tolist():
    i = len(cd_hit_map)
    cd_hit_map[i] = [seq, 'test_fun']

#####
for seq in train_non_func_predict_subseqs:
    i = len(cd_hit_map)
    cd_hit_map[i] = [seq, 'train_predict_nonfun']

for seq in val_non_func_predict_subseqs:
    i = len(cd_hit_map)
    cd_hit_map[i] = [seq, 'val_predict_nonfun']

for seq in test_non_func_predict_subseqs:
    i = len(cd_hit_map)
    cd_hit_map[i] = [seq, 'test_predict_nonfun']

with open('cd_hit_input.fasta', 'w') as f:
    for id, val in cd_hit_map.items():
        f.write(f">{id}_id\n{val[0]}\n")

In [119]:
import subprocess
cmd = "cd-hit -i cd_hit_input.fasta -o cd_hit_output.fasta -c 0.7 -n 5 -l 5 -d 0"
subprocess.run(cmd, shell=True, check=True)

print("CD-HIT clustering complete.")

Program: CD-HIT, V4.8.1 (+OpenMP), Nov 12 2024, 10:35:24
Command: cd-hit -i cd_hit_input.fasta -o cd_hit_output.fasta
         -c 0.7 -n 5 -l 5 -d 0

Started: Mon Aug 17 09:25:14 2026
                            Output                              
----------------------------------------------------------------
total seq: 849000
longest and shortest : 1351 and 6
Total letters: 33028516
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 138M
Buffer          : 1 X 26M = 26M
Table           : 1 X 78M = 78M
Miscellaneous   : 10M
Total           : 254M

Table limit with the given memory limit:
Max number of representatives: 3422988
Max number of word counting entries: 68182077

comparing sequences from          0  to     849000
..........    10000  finished       7601  clusters
..........    20000  finished      13648  clusters
..........    30000  finished      18948  clusters
..........    40000  finished      23615  clusters
..........    50000  finis

In [120]:
# reading clusters
clusters = []
with open("cd_hit_output.fasta.clstr") as f:
    for line in f:
        if line.startswith(">"):
            clusters.append([])
        else:
            key = int(line.split('>')[1].split('_id')[0].strip())

            clusters[-1].append(cd_hit_map[key])


In [121]:
# filturing clusters

# removing clusters having functional seq
new_clusters = []
for cluster in clusters:
    keep = True
    for seq in cluster:
        if '_fun' in seq[1]:
            keep = False
    
    if keep:
        new_clusters.append(cluster)

In [122]:
train_non_func_predict_subseqs_after_cd_hit = []
val_non_func_predict_subseqs_after_cd_hit = []
test_non_func_predict_subseqs_after_cd_hit = []

for cluster in new_clusters:
    classes = dict()
    for seq in cluster:
        classes[seq[1]] = seq[0]
    
    c = random.choice(list(classes.keys()))
    if 'train_predict_nonfun' == c:
        train_non_func_predict_subseqs_after_cd_hit.append(classes['train_predict_nonfun'])
    elif 'val_predict_nonfun' == c:
        val_non_func_predict_subseqs_after_cd_hit.append(classes['val_predict_nonfun'])
    elif 'test_predict_nonfun' == c:
        test_non_func_predict_subseqs_after_cd_hit.append(classes['test_predict_nonfun'])
    else:
        print('sss')


In [123]:
print('train pseudo non func: ', len(train_non_func_predict_subseqs), '-->', len(train_non_func_predict_subseqs_after_cd_hit))
print('val pseudo non func: ', len(val_non_func_predict_subseqs), '-->', len(val_non_func_predict_subseqs_after_cd_hit))
print('test pseudo non func: ', len(test_non_func_predict_subseqs), '-->', len(test_non_func_predict_subseqs_after_cd_hit))

train pseudo non func:  526873 --> 104446
val pseudo non func:  75816 --> 16627
test pseudo non func:  151614 --> 32303


In [124]:
# saving
with open("train_non_func_predict_subseqs_after_cd_hit.txt", 'w') as f:
    for seq in train_non_func_predict_subseqs_after_cd_hit:
        f.write(seq + '\n')

with open("val_non_func_predict_subseqs_after_cd_hit.txt", 'w') as f:
    for seq in val_non_func_predict_subseqs_after_cd_hit:
        f.write(seq + '\n')

with open("test_non_func_predict_subseqs_after_cd_hit.txt", 'w') as f:
    for seq in test_non_func_predict_subseqs_after_cd_hit:
        f.write(seq + '\n')

# retraining network

In [125]:
class PeptideDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        label_columns,
        max_length: int = 128,
        non_func_skip= 1

    ):
        """
        Args:
            dataframe (pd.DataFrame): DataFrame containing the data.
                                      Must have a 'sequence' column.
            tokenizer (AutoTokenizer): A Hugging Face tokenizer (e.g., for ESM-2).
            label_columns (List[str]): A list of column names that represent the labels.
            max_length (int): Maximum sequence length for padding/truncation.
        """
        self.df = dataframe
        self.tokenizer = tokenizer
        self.sequences = self.df['sequence'].values
        self.labels = self.df[label_columns].values
        self.label_columns = label_columns
        self.max_length = max_length

        functional_idxs = []
        non_functional_idxs = []
        for i, label in enumerate(self.labels):
            if label[endpoint_index['non-functional']] == True:
                non_functional_idxs.append(i)
            else:
                functional_idxs.append(i)
        
        non_functional_idxs = [non_functional_idxs[i: i+non_func_skip] for i in range(0, len(non_functional_idxs), non_func_skip)]

        functional_idxs = [functional_idxs[i: i+1] for i in range(0, len(functional_idxs), 1)]
        self.all_idxs = non_functional_idxs + functional_idxs

        # Pre-tokenize ALL sequences once
        sequences = dataframe['sequence'].tolist()
        encodings = tokenizer(
            sequences,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        self.input_ids = encodings['input_ids']
        self.attention_masks = encodings['attention_mask']


    def __len__(self) -> int:
        """Returns the total number of samples in the dataset."""
        return len(self.all_idxs)

    def __getitem__(self, index: int):
        """
        Retrieves a single sample from the dataset.

        Args:
            index (int): The index of the sample to retrieve.

        Returns:
            A dictionary containing:
            - 'input_ids': Token IDs of the sequence.
            - 'attention_mask': Mask to avoid performing attention on padding tokens.
            - 'labels': A multi-hot encoded tensor of labels.
            - 'img_input': A dummy tensor to match the MultiModelNetwork's forward signature.
        """
        idx = random.choice(self.all_idxs[index])

        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'labels': torch.FloatTensor(self.labels[idx])
        }

In [126]:
# reading non-functional sequences
train_non_func_predict_subseqs_after_cd_hit = []
val_non_func_predict_subseqs_after_cd_hit = []
test_non_func_predict_subseqs_after_cd_hit = []

with open("train_non_func_predict_subseqs_after_cd_hit.txt") as f:
    for seq in f.readlines():
        train_non_func_predict_subseqs_after_cd_hit.append(seq.strip())

with open("val_non_func_predict_subseqs_after_cd_hit.txt") as f:
    for seq in f.readlines():
        val_non_func_predict_subseqs_after_cd_hit.append(seq.strip())

with open("test_non_func_predict_subseqs_after_cd_hit.txt") as f:
    for seq in f.readlines():
        test_non_func_predict_subseqs_after_cd_hit.append(seq.strip())

In [127]:
new_train_rows = []
for seq in train_non_func_predict_subseqs_after_cd_hit:
    row = {endpoint: False for endpoint in endpoints}
    row['non-functional'] = True
    row['sequence'] = seq
    new_train_rows.append(row)

new_val_rows = []
for seq in val_non_func_predict_subseqs_after_cd_hit:
    row = {endpoint: False for endpoint in endpoints}
    row['non-functional'] = True
    row['sequence'] = seq
    new_val_rows.append(row)

new_test_rows = []
for seq in test_non_func_predict_subseqs_after_cd_hit:
    row = {endpoint: False for endpoint in endpoints}
    row['non-functional'] = True
    row['sequence'] = seq
    new_test_rows.append(row)

new_train_df = pd.DataFrame(new_train_rows)
new_val_df = pd.DataFrame(new_val_rows)
new_test_df = pd.DataFrame(new_test_rows)

In [128]:
new_train_df = pd.concat([train_df, new_train_df], ignore_index=True)
new_val_df = pd.concat([val_df, new_val_df], ignore_index=True)
new_test_df = pd.concat([test_df, new_test_df], ignore_index=True)

In [129]:
LABEL_COLUMNS = endpoints

MAX_LEN = 100
train_dataset = PeptideDataset(new_train_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN, non_func_skip=1)
val_dataset = PeptideDataset(new_val_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN, non_func_skip=1)
test_dataset = PeptideDataset(new_test_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN, non_func_skip=1)

In [131]:
print(len(train_dataset), len(val_dataset), len(test_dataset))

284917 42408 83863


In [132]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [133]:
import copy
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        # Update parameters (weights & biases)
        for s_param, m_param in zip(self.shadow.parameters(), model.parameters()):
            s_param.data.mul_(self.decay).add_(m_param.data, alpha=1 - self.decay)
        
        # FIX: Update only floating point buffers (skip int buffers like num_batches_tracked)
        for s_buf, m_buf in zip(self.shadow.buffers(), model.buffers()):
            if s_buf.dtype.is_floating_point:
                s_buf.data.mul_(self.decay).add_(m_buf.data, alpha=1 - self.decay)
            else:
                # For integer buffers, just copy directly (no EMA smoothing needed)
                s_buf.data.copy_(m_buf.data)

    def forward(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)

In [134]:
# calculate weight of non-func here
n_non_func = int(new_train_df["non-functional"].sum())
n_func = int(len(new_train_df) - n_non_func)

# Effective non-functional count per epoch after non_func_skip grouping
non_func_groups = max(len(train_dataset) - n_func, 1)

# For BCEWithLogitsLoss, positive class is non-functional
pos_weight_val = n_func / non_func_groups
pos_weight_tensor = torch.tensor([pos_weight_val], dtype=torch.float32, device=DEVICE)

print(f"non-functional: {n_non_func} | functional: {n_func}")
print(f"effective non-functional per epoch: {non_func_groups}")
print(f"pos_weight for BCE: {pos_weight_val:.4f}")

non-functional: 218239 | functional: 66678
effective non-functional per epoch: 218239
pos_weight for BCE: 0.3055


In [136]:
del id

In [141]:
torch.cuda.empty_cache()

# loading classification model
MODEL_PATH = 'phase_1_model.pt'

model = PeptideNetwork(num_classes=13, mask_token_id=32)
if os.path.exists(MODEL_PATH):
    print(f"Found {MODEL_PATH} ! Loading weights.")
    state_dict = torch.load(MODEL_PATH, map_location="cpu")
else:
    raise("Wrong model path")

esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE)

ema = EMA(model, decay=0.999)


# --- ASL: Asymmetric Loss for Multi-Label Classification ---
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        pos_part = targets * torch.log(p.clamp(min=1e-8))
        neg_p = (1 - p).clamp(min=1e-8)
        if self.clip > 0:
            neg_p = (neg_p + self.clip).clamp(max=1)
        neg_part = (1 - targets) * torch.log(neg_p)
        pos_weight = (1 - p) ** self.gamma_pos
        neg_weight = p ** self.gamma_neg
        loss = -(pos_weight * pos_part + neg_weight * neg_part)
        return loss.mean()

criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=2, clip=0.03)
binary_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val))#

# Optimizer — 3 LR groups (Lowered LRs to prevent early overfitting)
esm_t6_ids = set(id(p) for p in model.esm_t6_a.esm_mlm.parameters())
esm_t6_b_ids = set(id(p) for p in model.esm_t6_b.esm_mlm.parameters())

esm_t6_params = [p for p in model.parameters() if id(p) in esm_t6_ids and p.requires_grad]
esm_t6_b_params = [p for p in model.parameters() if id(p) in esm_t6_b_ids and p.requires_grad]
other_params = [p for p in model.parameters() if id(p) not in esm_t6_ids and id(p) not in esm_t6_b_ids and p.requires_grad]

optimizer = optim.AdamW([
    {'params': esm_t6_params, 'lr': 1e-6},     # Reduced from 1e-5
    {'params': esm_t6_b_params, 'lr': 2e-7},   # Reduced from 2e-6
    {'params': other_params, 'lr': 1e-5},      # Reduced from 1e-4
], weight_decay=0.05)                          # Kept the stronger L2 regularization

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {total_params/1e6:.1f}M | Trainable: {trainable_params/1e6:.5f}M")
print(f"Loss: ASL(γ-=4, γ+=1, clip=0.05) | v19: Triple CrossAttn + Task Query Decoder")

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Found phase_1_model.pt ! Loading weights.
Total: 18.7M | Trainable: 13.61230M
Loss: ASL(γ-=4, γ+=1, clip=0.05) | v19: Triple CrossAttn + Task Query Decoder


In [142]:
# this part is from earler training
# Warmup + Cosine Annealing
epochs = 40
steps_per_epoch = len(train_loader)
warmup_epochs = 1
total_steps = epochs * steps_per_epoch

def lr_lambda(step):
    if step < warmup_epochs * steps_per_epoch:
        return step / (warmup_epochs * steps_per_epoch)
    progress = (step - warmup_epochs * steps_per_epoch) / (total_steps - warmup_epochs * steps_per_epoch)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"Warmup ({warmup_epochs} ep) + CosineAnnealing, total {epochs} epochs, {total_steps} steps")


Warmup (1 ep) + CosineAnnealing, total 40 epochs, 178080 steps


In [143]:
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import json

In [144]:
# --- HELPER FUNCTIONS ---
# training without gate
def calculate_complete_metrics(y_true, y_pred_probs, thresholds, class_names):
    results = {}
    for i in range(y_true.shape[1]):
        y_pred_cls = (y_pred_probs[:, i] >= thresholds[i]).astype(int)
        mcc = matthews_corrcoef(y_true[:, i], y_pred_cls)
        acc = accuracy_score(y_true[:, i], y_pred_cls)
        prec = precision_score(y_true[:, i], y_pred_cls, zero_division=0)
        rec = recall_score(y_true[:, i], y_pred_cls, zero_division=0)
        f1 = f1_score(y_true[:, i], y_pred_cls, zero_division=0)
        try:
            auc_val = roc_auc_score(y_true[:, i], y_pred_probs[:, i])
        except ValueError:
            auc_val = 0.5
        tn, fp, fn, tp = confusion_matrix(y_true[:, i], y_pred_cls, labels=[0, 1]).ravel()
        results[class_names[i]] = {
            "mcc": float(mcc),
            "threshold": float(thresholds[i]),
            "accuracy": float(acc),
            "precision": float(prec),
            "recall": float(rec),
            "f1_score": float(f1),
            "roc_auc": float(auc_val),
            "tp": int(tp),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
        }
    return results

def find_optimal_thresholds(y_true, y_pred_probs):
    optimal_thresholds = []
    for i in range(y_true.shape[1]):
        best_mcc = -1
        best_threshold = 0.5
        for threshold in np.arange(0.1, 0.95, 0.05):
            y_pred_class = (y_pred_probs[:, i] >= threshold).astype(int)
            mcc = matthews_corrcoef(y_true[:, i], y_pred_class)
            if mcc > best_mcc:
                best_mcc = mcc
                best_threshold = threshold
        optimal_thresholds.append(best_threshold)
    return optimal_thresholds

def calculate_mcc(tp, tn, fp, fn):
    num = (tp * tn) - (fp * fn)
    den_sq = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    return num / math.sqrt(den_sq) if den_sq > 0 else 0.0

def aggregate_mcc(*metric_groups):
    tp, tn, fp, fn = 0, 0, 0, 0
    for metrics in metric_groups:
        for values in metrics.values():
            tp += values['tp']
            tn += values['tn']
            fp += values['fp']
            fn += values['fn']
    return calculate_mcc(tp, tn, fp, fn)

def feature_mixup(features, labels, alpha=0.4):
    batch_size = features.size(0)
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(batch_size, device=features.device)
    mixed_features = lam * features + (1 - lam) * features[index]
    mixed_labels = lam * labels + (1 - lam) * labels[index]
    return mixed_features, mixed_labels

def rdrop_kl_loss(logits1, logits2):
    kl1 = F.binary_cross_entropy_with_logits(logits1, torch.sigmoid(logits2).detach(), reduction='mean')
    kl2 = F.binary_cross_entropy_with_logits(logits2, torch.sigmoid(logits1).detach(), reduction='mean')
    return (kl1 + kl2) / 2

def enable_dropout(model):
    """
    Specifically tailored for PeptideNetwork to activate all sources of dropout
    during Test-Time Augmentation (TTA), including functional dropouts hidden 
    inside complex PyTorch modules.
    """
    for module in model.modules():
        class_name = module.__class__.__name__
        
        # 1. Standard explicit dropout layers (Dropout, Dropout1d, Dropout2d)
        if class_name.startswith('Dropout'):
            module.train()
            
        # 2. Cross-Attention functional dropout
        elif class_name == 'MultiheadAttention':
            module.train()
            
        # 3. Task Query Decoder functional dropout
        elif class_name in ['TransformerDecoder', 'TransformerDecoderLayer']:
            module.train()
            
        # 4. GRU functional dropout (applied between internal layers)
        elif class_name == 'GRU':
            module.train()


# --- SWA ---
swa_start_epoch = 15
swa_model = None
swa_n = 0

def update_swa(model_state, swa_state, n):
    if swa_state is None:
        return {k: v.clone() for k, v in model_state.items()}, 1
    for k in swa_state:
        swa_state[k] = (swa_state[k] * n + model_state[k]) / (n + 1)
    return swa_state, n + 1

# --- MAIN TRAINING LOOP ---
history_log = []
history_log_path = "phase_2_model.json"
best_model_path = "phase_2_model.pt"
best_val_mcc = -1.0
scaler = torch.amp.GradScaler('cuda')
patience_counter = 0
early_stop_patience = 10
mixup_alpha = 0.4
rdrop_alpha = 1.0
tta_passes = 5

func_names = [index_endpoint[i] for i in FUNC_INDICES]
bin_names = ["NON-FUNCTIONAL"]
bin_thresh = [0.5]
spec_thresh = [0.5] * len(func_names)

print(f"  R-Drop α={rdrop_alpha} | Mixup α={mixup_alpha} | SWA from ep {swa_start_epoch}")
print(f"  Validation eval: TTA={tta_passes} | Test eval: TTA={tta_passes}")
print("=" * 70)
for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0

    for data in train_loader:
        input_ids = data['input_ids'].to(DEVICE)
        attention_mask = data['attention_mask'].to(DEVICE)
        labels = data['labels'].to(DEVICE)

        optimizer.zero_grad()

        # Binary label: 1 if NON-FUNCTIONAL, 0 if FUNCTIONAL
        target_bin = labels[:, NON_FUNC_IDX].unsqueeze(1).float()
        target_spec = labels[:, FUNC_INDICES]

        with torch.amp.autocast('cuda'):
            # Two augmented passes for R-Drop
            bin_features1, features1 = model.get_features(input_ids, attention_mask, mask_tokens=True)
            bin_features2, features2 = model.get_features(input_ids, attention_mask, mask_tokens=True)

            # 1. Clean predictions
            logits1_clean = model.multi_classify(features1)
            logits2_clean = model.multi_classify(features2)

            # 2. Binary classification (Always on the whole batch)
            binary_logits = model.binary_classify(bin_features2)
            binary_loss = binary_criterion(binary_logits, target_bin)
            ortho = model.ortho_loss()

            # multi head on whole
            func_f1 = features1
            func_targets = target_spec
            func_logits1 = logits1_clean
            func_logits2 = logits2_clean

            # R-Drop only on functional
            rdrop_loss = rdrop_kl_loss(func_logits1, func_logits2)
            
            # Mixup only on functional
            mixed_f, mixed_y = feature_mixup(func_f1, func_targets, alpha=mixup_alpha)
            logits_mix = model.multi_classify(mixed_f)
            
            # ASL evaluates ONLY the functional slice
            asl_loss_mix = criterion(logits_mix, mixed_y)
            asl_loss_clean = criterion(func_logits2, func_targets)
            
            spec_loss = (asl_loss_mix + asl_loss_clean) / 2 + (rdrop_alpha * rdrop_loss) + (0.1 * ortho)
            loss = 0.3 * binary_loss + 0.7 * spec_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema.update(model)

        total_train_loss += loss.item() * input_ids.size(0)

    # --- SWA Update ---
    if epoch >= swa_start_epoch:
        swa_model, swa_n = update_swa(ema.shadow.state_dict(), swa_model, swa_n)

    if swa_model is not None and epoch >= swa_start_epoch:
        eval_model = copy.deepcopy(ema.shadow)
        eval_model.load_state_dict(swa_model)
        eval_model.eval()
    else:
        eval_model = ema.shadow
        eval_model.eval()

    # --- VALIDATION (TTA) ---
    total_val_loss = 0.0
    val_bin_y = []
    val_spec_y = []
    val_tta_bin_preds = [[] for _ in range(tta_passes)]
    val_tta_spec_preds = [[] for _ in range(tta_passes)]

    with torch.no_grad():
        for data in val_loader:
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            labels = data['labels'].to(DEVICE)

            target_bin = labels[:, NON_FUNC_IDX].unsqueeze(1).float()
            target_spec = labels[:, FUNC_INDICES]

            with torch.amp.autocast('cuda'):
                pred_bin, pred_spec = eval_model(input_ids, attention_mask)
                loss_bin = binary_criterion(pred_bin, target_bin)

                is_functional_mask = (target_bin == 0).view(-1).bool()
                if is_functional_mask.sum() > 0:
                    loss_spec = criterion(pred_spec[is_functional_mask], target_spec[is_functional_mask])
                    loss_val = 0.3 * loss_bin + 0.7 * loss_spec
                else:
                    loss_val = loss_bin  # fallback if batch has no functional peptides

            total_val_loss += loss_val.item() * input_ids.size(0)
            
            # Append entire batch for both binary and specific targets
            val_bin_y.append(target_bin.cpu().numpy())
            val_spec_y.append(target_spec.cpu().numpy()) 
            
            val_tta_bin_preds[0].append(torch.sigmoid(pred_bin).detach().cpu().numpy())
            val_tta_spec_preds[0].append(torch.sigmoid(pred_spec).detach().cpu().numpy())

            if tta_passes > 1:
                enable_dropout(eval_model)
                for t in range(1, tta_passes):
                    with torch.amp.autocast('cuda'):
                        pred_bin_t, pred_spec_t = eval_model(input_ids, attention_mask)
                    val_tta_bin_preds[t].append(torch.sigmoid(pred_bin_t).detach().cpu().numpy())
                    val_tta_spec_preds[t].append(torch.sigmoid(pred_spec_t).detach().cpu().numpy())
                eval_model.eval()

    y_true_bin_val = np.concatenate(val_bin_y)
    y_pred_bin_val = np.mean([np.concatenate(pred_list) for pred_list in val_tta_bin_preds], axis=0)
    
    y_true_spec_val = np.concatenate(val_spec_y)
    raw_pred_spec_val = np.mean([np.concatenate(pred_list) for pred_list in val_tta_spec_preds], axis=0)

    # SOFT GATING: Scale specific predictions by functionality probability
    # y_pred_bin_val is probability of NON-FUNCTIONAL, so 1 - y_pred_bin_val is prob of FUNCTIONAL
    y_pred_spec_val = raw_pred_spec_val * (1 - y_pred_bin_val)

    # Compute optimal thresholds on Validation set
    opt_bin_thresh = find_optimal_thresholds(y_true_bin_val.reshape(-1, 1), y_pred_bin_val.reshape(-1, 1))
    opt_spec_thresh = find_optimal_thresholds(y_true_spec_val, raw_pred_spec_val)


    val_bin_metrics = calculate_complete_metrics(y_true_bin_val, y_pred_bin_val, opt_bin_thresh, bin_names)
    val_spec_metrics = calculate_complete_metrics(y_true_spec_val, y_pred_spec_val, opt_spec_thresh, func_names)
    val_total_mcc = aggregate_mcc(val_bin_metrics, val_spec_metrics)
    
    val_func_mccs = [metrics['mcc'] for metrics in val_spec_metrics.values() if not np.isnan(metrics['mcc'])]
    val_avg_func_mcc = sum(val_func_mccs) / len(val_func_mccs) if val_func_mccs else 0.0


    # --- TEST (TTA) ---
    total_test_loss = 0.0
    test_bin_y = []
    test_spec_y = []
    test_tta_bin_preds = [[] for _ in range(tta_passes)]
    test_tta_spec_preds = [[] for _ in range(tta_passes)]

    with torch.no_grad():
        for data in test_loader:
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            labels = data['labels'].to(DEVICE)

            target_bin = labels[:, NON_FUNC_IDX].unsqueeze(1).float()
            target_spec = labels[:, FUNC_INDICES]

            with torch.amp.autocast('cuda'):
                pred_bin, pred_spec = eval_model(input_ids, attention_mask)
                loss_bin = binary_criterion(pred_bin, target_bin)
                
                is_functional_mask = (target_bin == 0).view(-1).bool()
                if is_functional_mask.sum() > 0:
                    loss_spec = criterion(pred_spec[is_functional_mask], target_spec[is_functional_mask])
                    loss_test = 0.3 * loss_bin + 0.7 * loss_spec
                else:
                    loss_test = loss_bin

            total_test_loss += loss_test.item() * input_ids.size(0)
            
            test_bin_y.append(target_bin.cpu().numpy())
            test_spec_y.append(target_spec.cpu().numpy())
            
            # First pass predictions
            test_tta_bin_preds[0].append(torch.sigmoid(pred_bin).detach().cpu().numpy())
            test_tta_spec_preds[0].append(torch.sigmoid(pred_spec).detach().cpu().numpy())

            # Additional TTA passes
            if tta_passes > 1:
                enable_dropout(eval_model)
                for t in range(1, tta_passes):
                    with torch.amp.autocast('cuda'):
                        pred_bin_t, pred_spec_t = eval_model(input_ids, attention_mask)
                    test_tta_bin_preds[t].append(torch.sigmoid(pred_bin_t).detach().cpu().numpy())
                    test_tta_spec_preds[t].append(torch.sigmoid(pred_spec_t).detach().cpu().numpy())
                eval_model.eval()

    y_true_bin_test = np.concatenate(test_bin_y)
    y_pred_bin_test = np.mean([np.concatenate(pred_list) for pred_list in test_tta_bin_preds], axis=0)
    
    y_true_spec_test = np.concatenate(test_spec_y)
    raw_pred_spec_test = np.mean([np.concatenate(pred_list) for pred_list in test_tta_spec_preds], axis=0)

    # SOFT GATING for Test
    y_pred_spec_test = raw_pred_spec_test * (1 - y_pred_bin_test)

    test_bin_metrics = calculate_complete_metrics(y_true_bin_test, y_pred_bin_test, opt_bin_thresh, bin_names)
    test_spec_metrics = calculate_complete_metrics(y_true_spec_test, y_pred_spec_test, opt_spec_thresh, func_names)
    test_total_mcc = aggregate_mcc(test_bin_metrics, test_spec_metrics)

    test_func_mccs = [metrics['mcc'] for metrics in test_spec_metrics.values() if not np.isnan(metrics['mcc'])]
    test_avg_func_mcc = sum(test_func_mccs) / len(test_func_mccs) if test_func_mccs else 0.0

    avg_train_loss = total_train_loss / len(train_loader.dataset)
    avg_val_loss = total_val_loss / len(val_loader.dataset)
    avg_test_loss = total_test_loss / len(test_loader.dataset)

    if val_avg_func_mcc > best_val_mcc:
        best_val_mcc = val_avg_func_mcc
        patience_counter = 0
        torch.save(eval_model.state_dict(), best_model_path)
        marker = " >>> BEST <<<"
    else:
        patience_counter += 1
        marker = ""

    lr_now = optimizer.param_groups[0]['lr']
    swa_tag = f" [SWA n={swa_n}]" if epoch >= swa_start_epoch else ""
    print(f"Ep {epoch:02d}/{epochs} | LR: {lr_now:.1e} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | Test: {avg_test_loss:.4f}{swa_tag}")
    print(f"  MCC -> Func Avg (Val): {val_avg_func_mcc:.4f} | Func Avg (Test): {test_avg_func_mcc:.4f} | Pat: {patience_counter}/{early_stop_patience}{marker}")
    print(f"  Overall Total MCC -> Val: {val_total_mcc:.4f} | Test: {test_total_mcc:.4f}")
    print(f"  NON-FUNCTIONAL MCC -> Val: {val_bin_metrics['NON-FUNCTIONAL']['mcc']:.4f} | Test: {test_bin_metrics['NON-FUNCTIONAL']['mcc']:.4f}")
    
    for name in func_names:
        val_mcc = val_spec_metrics[name]['mcc'] if name in val_spec_metrics else float('nan')
        test_mcc = test_spec_metrics[name]['mcc'] if name in test_spec_metrics else float('nan')
        print(f"  {name:16s}: Val MCC={val_mcc:.4f} | Test MCC={test_mcc:.4f}")
    print("-" * 70)

    epoch_data = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "test_loss": avg_test_loss,
        "val_avg_func_mcc": val_avg_func_mcc,
        "val_total_mcc": float(val_total_mcc),
        "test_total_mcc": float(test_total_mcc),
        "test_avg_func_mcc": float(test_avg_func_mcc),
        "thresholds": {
            "binary": opt_bin_thresh,
            "functional": opt_spec_thresh,
        },
        "val_metrics": {**val_bin_metrics, **val_spec_metrics},
        "test_metrics": {**test_bin_metrics, **test_spec_metrics},
    }
    history_log.append(epoch_data)
    with open(history_log_path, "w") as f:
        json.dump(history_log, f, indent=2)

    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping at epoch {epoch}!")
        break

    if swa_model is not None and epoch >= swa_start_epoch:
        del eval_model

print(f"\nTraining Complete! Best Val MCC: {best_val_mcc:.4f}")

  R-Drop α=1.0 | Mixup α=0.4 | SWA from ep 15
  Validation eval: TTA=5 | Test eval: TTA=5
Ep 01/40 | LR: 1.0e-06 | Train: 0.4479 | Val: 0.0643 | Test: 0.0612
  MCC -> Func Avg (Val): 0.5628 | Func Avg (Test): 0.5750 | Pat: 0/10 >>> BEST <<<
  Overall Total MCC -> Val: 0.8698 | Test: 0.8686
  NON-FUNCTIONAL MCC -> Val: 0.7750 | Test: 0.7867
  anti-bacterial  : Val MCC=0.7752 | Test MCC=0.8259
  anti-cancer     : Val MCC=0.5283 | Test MCC=0.5257
  anti-fungal     : Val MCC=0.5870 | Test MCC=0.5669
  anti-parasitic  : Val MCC=0.5353 | Test MCC=0.5506
  anti-viral      : Val MCC=0.3985 | Test MCC=0.5025
  cell-cell-communication: Val MCC=0.6100 | Test MCC=0.6553
  drug-delivery   : Val MCC=0.4808 | Test MCC=0.4264
  immunological   : Val MCC=0.4410 | Test MCC=0.4632
  inhibitor       : Val MCC=0.4230 | Test MCC=0.3296
  metabolic       : Val MCC=0.6057 | Test MCC=0.6012
  other-functional: Val MCC=0.3001 | Test MCC=0.4518
  signal-peptide  : Val MCC=0.8591 | Test MCC=0.8551
  toxic        